In [1]:
from dotenv import load_dotenv
import os
import vertexai
from vertexai.generative_models import GenerativeModel
load_dotenv()
project_id = "ai-signposting"
location="europe-west2"
vertexai.init(project=project_id, location=location)
model = GenerativeModel(model_name="gemini-1.5-flash-001") 
#get the database
mongodb_uri=os.environ.get("MONGO_URI")


In [2]:
from pymongo import MongoClient
client=MongoClient(mongodb_uri)
db=client["signposting_db"]

We can try this with the same input as with the previous file first:

In [4]:
def get_tagged_options(tag, location_choice, collection_name):
    collection=db[collection_name]
    formatted_tag="#"+tag
    options=collection.find({"Category tags": formatted_tag, "Local / National":location_choice}, projection={"_id":0})
    return list(options)
tag="benefits-advice"
location_choice="National"
collection_name="support_options"
options=get_tagged_options(tag, location_choice, collection_name)
for idx, option in enumerate(options):
    print(f"Option {idx+1}: {option}")

Option 1: {'Category tags': ['#debt-advice', '#budget-advice', '#benefits-advice'], 'Name': 'Step Change', 'Website': 'https://www.stepchange.org/', 'Phone - call': '0800 138 1111', 'Local / National': 'National', 'Postcode': '', 'Prefix': '', 'Short text description': 'Offers free comprehensive debt advice to support individuals gain financial control and stabilit', 'Logo-link': 'https://drive.google.com/uc?export=download&id=1kKCOELT1W2_vNdG-zj3PXIDmJCDGnxKC', 'Email': '', 'latitude': None, 'longitude': None}
Option 2: {'Category tags': ['#debt-advice', '#budget-advice', '#benefits-advice'], 'Name': 'Money Helper', 'Website': 'https://www.moneyhelper.org.uk/en', 'Phone - call': '', 'Local / National': 'National', 'Postcode': '', 'Prefix': '', 'Short text description': 'Offers free and impartial help with money and pensions, supported by the government', 'Logo-link': 'https://drive.google.com/uc?export=download&id=1IS934FbUi4zv8mE1GBvlSzgq70X54Pw_', 'Email': '', 'latitude': None, 'lon

We would probably extract just the relevant info prior to feeding each organization info object into the model:

In [7]:
def create_input_text(option):
    website=option["Website"]
    if option["Local / National"]=="National":
        location=option["Local / National"]
    else:
        location=option["Postcode"]
    description=option["Short text description"]
    name=option["Name"]
    result={"website": website, "location": location, "description": description, "name": name}
    return result
option_summaries=[create_input_text(option) for option in options]
print(option_summaries)

[{'website': 'https://www.stepchange.org/', 'location': 'National', 'description': 'Offers free comprehensive debt advice to support individuals gain financial control and stabilit', 'name': 'Step Change'}, {'website': 'https://www.moneyhelper.org.uk/en', 'location': 'National', 'description': 'Offers free and impartial help with money and pensions, supported by the government', 'name': 'Money Helper'}, {'website': 'https://capuk.org/', 'location': 'National', 'description': 'Offers free, expert debt help to support individual’s financial recovery and stability', 'name': 'Christians Against Poverty'}, {'website': 'https://www.uceplus.co.uk/about', 'location': 'National', 'description': 'Charity that aims to make essential financial information and support clear and accessible to the public', 'name': 'Universal Credit Essentials'}, {'website': 'https://www.moneywellness.com/debt-advice', 'location': 'National', 'description': 'Offers free financial guidance, promoting well-being through

Now we can make a prompt, first to just write a free form message with all relevant info (if sorting by postcode we would sort first, translation would be the last step most likely)

In [9]:
def describe_organization(option, model):
    prompt=f"""
    This is a dictionary representing information on a support organization within the UK. 
    Using all of the information, write a description of the organization. 
    Make sure all details written in the dictionary are included. 
    Mention the name first. Include any additional details if you have knowledge of them. Keep your answer in the range of 3 sentences.
    Organization dictionary:
    {option}
    """
    response=model.generate_content(prompt) #A LOT we can configure, customize and set up here, probably not necessary for now though
    print(response.text)
    print("\n")

In [11]:
import json
inputs=[json.dumps(option) for option in option_summaries]
model=GenerativeModel(model_name="gemini-1.5-flash-001")
for input_item in inputs:
    describe_organization(input_item, model)

Step Change is a national organization that provides free, comprehensive debt advice across the UK.  Their mission is to help individuals gain financial control and stability by offering support and guidance to navigate their debt situation.  Their website, https://www.stepchange.org/, is a valuable resource for anyone struggling with debt, providing information, tools, and access to their expert advisors. 



Money Helper is a national organization offering free and impartial advice on money and pensions.  They are supported by the UK government and can be reached through their website at https://www.moneyhelper.org.uk/en.  Money Helper is a valuable resource for individuals seeking guidance on managing their finances and planning for retirement. 



Christians Against Poverty (CAP) is a national organization based in the UK that provides free, expert debt help to individuals across the country.  Their mission is to support individuals' financial recovery and stability, empowering the